# F04 EDICTA — PALATIUM
> *Frégate d'Encodage Final — Séquence PNG → MP4 4K*

**Pipeline :**
1. Monter Drive + valider IN/
2. Installer FFmpeg
3. Encoder via pipeline ou monitor
4. Télécharger les MP4
5. Valider avec L'Arbitre

---
**Doctrine :** F04 ne lit que `F04_EDICTA/IN/` et n'écrit que dans `F04_EDICTA/OUT_FINAL/`.

In [ ]:
# ─── CONFIGURATION ─────────────────────────────────────────
DRIVE_ROOT = '/content/drive/MyDrive/DRIVE_PALATIUM'

# Formats à encoder : 'shorts', 'youtube', ou ['shorts', 'youtube'] pour les deux
RENDER_FORMATS = ['shorts', 'youtube']

# Mode : 'direct' (notebook) ou 'monitor' (interface Flask + ngrok)
MODE = 'direct'

FLASK_PORT  = 5004
NGROK_TOKEN = ''   # ← Votre token ngrok (mode 'monitor' uniquement)
# ───────────────────────────────────────────────────────────

## Étape 1 — Monter Drive + Installer dépendances

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!apt-get install -y ffmpeg -q
!pip install flask flask-cors pyngrok -q
!ffmpeg -version 2>&1 | head -1
print('✅ Setup complet')

## Étape 2 — Valider les fichiers IN/ (L'Arbitre CHECK-IN)

In [ ]:
import shutil, subprocess
shutil.copy(f'{DRIVE_ROOT}/PAL_ARBITRE.py', '/content/PAL_ARBITRE.py')

ret = subprocess.run(['python', '/content/PAL_ARBITRE.py',
    '--frigate', 'F04', '--mode', 'check-in',
    '--drive-root', DRIVE_ROOT, '--verbose'],
    capture_output=False)

if ret.returncode != 0:
    raise RuntimeError('F04 BLOQUÉE — Transférer les frames IN/OUT_FRAMES/ avant de continuer')
print('✅ F04 IN/ validé')

## Étape 3 — Copier les scripts

In [ ]:
import shutil
CODEBASE = f'{DRIVE_ROOT}/F04_EDICTA/CODEBASE'
for f in ['pal_f04_pipeline.py', 'pal_f04_flask.py', 'pal_f04_monitor.html']:
    shutil.copy(f'{CODEBASE}/{f}', f'/content/{f}')
    print(f'✅ {f}')

## Étape 4a — Mode DIRECT : Encoder depuis le notebook

In [ ]:
if MODE == 'direct':
    import sys
    sys.path.insert(0, '/content')
    from pal_f04_pipeline import run_f04_pipeline

    result = run_f04_pipeline(DRIVE_ROOT, RENDER_FORMATS)

    if result['success']:
        print('\n  ✅ Encodage terminé avec succès !')
    else:
        print('\n  ❌ Encodage échoué — voir messages ci-dessus')
else:
    print('Mode monitor sélectionné — exécuter la cellule 4b')

## Étape 4b — Mode MONITOR : Interface Flask + ngrok

In [ ]:
if MODE == 'monitor':
    import threading, time
    from pyngrok import ngrok, conf
    import sys
    sys.path.insert(0, '/content')

    if NGROK_TOKEN:
        conf.get_default().auth_token = NGROK_TOKEN

    from pal_f04_flask import app, init_config
    init_config(DRIVE_ROOT)

    t = threading.Thread(target=lambda: app.run(
        host='0.0.0.0', port=FLASK_PORT, debug=False, use_reloader=False), daemon=True)
    t.start()
    time.sleep(1)

    url = ngrok.connect(FLASK_PORT)
    print(f'\n  ✅ MONITOR DISPONIBLE ICI :')
    print(f'  🔗 {url}')
    print('\n  Cliquer « ENCODER » dans le monitor pour lancer l\'encodage.')
else:
    print('Mode direct sélectionné — exécuter la cellule 4a')

## Étape 5 — Télécharger les MP4

In [ ]:
from google.colab import files
from pathlib import Path

out_dir = Path(DRIVE_ROOT) / 'F04_EDICTA' / 'OUT_FINAL'

for fmt, name in [('shorts', 'shorts_vertical.mp4'), ('youtube', 'youtube_horizontal.mp4')]:
    if fmt in RENDER_FORMATS:
        p = out_dir / name
        if p.exists():
            size_mb = p.stat().st_size / 1024 / 1024
            print(f'✅ {name} ({size_mb:.1f} MB)')
            files.download(str(p))
        else:
            print(f'⚠️  {name} introuvable — encodage terminé ?')

## Étape 6 — Valider OUT_FINAL/ (L'Arbitre CHECK-OUT)

In [ ]:
import subprocess
ret = subprocess.run(['python', '/content/PAL_ARBITRE.py',
    '--frigate', 'F04', '--mode', 'check-out',
    '--drive-root', DRIVE_ROOT, '--verbose'],
    capture_output=False)

if ret.returncode == 0:
    print('\n  ✅ F04 EDICTA SCELLÉE — Campagne terminée')
    print('     MP4 disponibles pour le Magos dans OUT_FINAL/')
else:
    print('\n  ❌ Vérification échouée — voir messages ci-dessus')

---
## Campagne Terminée

```
Forge des Frégates : [██████████] 4/4 Frégates Scellées (100%)
Fleet Seal         : ACCORDÉ
Objectif           : 1ère villa rendue
```

**Inscrire dans `TRACKING/PALATIUM_CAMPAIGN_LOG.md` et `TRACKING/PALATIUM_TRANSFER_LOG.md`.**

*F04 EDICTA — Scellée. Fleet Seal accordé. Ad Victoriam.*